# Testing different optimizers


In [1]:
from src.cnn_utils import get_dataset

train_dataset, val_dataset = get_dataset()

# check dataset loaded correctly
print(f"Training dataset size: {len(train_dataset)}")
print(f"Validation dataset size: {len(val_dataset)}")

Training dataset size: 24000
Validation dataset size: 6000


In [2]:
import copy
import time
from dataclasses import dataclass, asdict
from typing import Dict, Tuple

import torch
import torch.nn as nn
from torch.utils.data import DataLoader

import wandb


# =========================================================
# CONFIG
# =========================================================

@dataclass
class Config:
    in_channels: int = 3
    num_classes: int = 10

    lr: float = 1e-3
    depth: int = 14                  # fixed depth for optimizer comparison
    inputsize: int = 224

    batch_size: int = 64
    epochs: int = 20

    base_channels: int = 32
    max_channels: int = 256
    dropout: float = 0.0            # dropout in conv blocks
    do: float = 0.5                 # dropout in classifier head

    num_workers: int = 0
    pin_memory: bool = True

    device: str = "cuda" if torch.cuda.is_available() else "cpu"

    optimizers: Tuple[str, ...] = ("SGD", "MomentumNesterov", "RMSprop", "Adam")

    # W&B
    use_wandb: bool = True
    wandb_project: str = "MPW-CNN"
    wandb_entity: str = "MSE_DeLearn_SPR26"
    wandb_mode: str = "online"      # "online", "offline", or "disabled"
    wandb_watch: bool = False
    wandb_log_freq: int = 100


# =========================================================
# MODEL
# =========================================================

class ConvBlock(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, use_pool: bool = False, dropout: float = 0.0):
        super().__init__()

        layers = [
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        ]

        if use_pool:
            layers.append(nn.MaxPool2d(kernel_size=2, stride=2))

        if dropout > 0:
            layers.append(nn.Dropout2d(dropout))

        self.block = nn.Sequential(*layers)

    def forward(self, x):
        return self.block(x)


class DepthCNN(nn.Module):
    def __init__(
        self,
        depth: int,
        in_channels: int = 3,
        num_classes: int = 10,
        base_channels: int = 32,
        max_channels: int = 256,
        dropout: float = 0.0,
        do: float = 0.5,
        inputsize: int = 224,
    ):
        super().__init__()

        layers = []

        current_in = in_channels
        current_out = base_channels
        current_size = inputsize
        pool_count = 0

        for i in range(depth):
            want_pool = ((i + 1) % 2 == 0)

            # limit pooling so feature maps do not become too small
            use_pool = want_pool and current_size >= 2 and pool_count < 4

            layers.append(
                ConvBlock(
                    in_channels=current_in,
                    out_channels=current_out,
                    use_pool=use_pool,
                    dropout=dropout if depth >= 4 else 0.0,
                )
            )

            current_in = current_out

            if use_pool:
                pool_count += 1
                current_size = current_size // 2 #
                current_out = min(current_out * 2, max_channels)

        self.features = nn.Sequential(*layers)

        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.Dropout(p=do),
            nn.Linear(current_in, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x


# =========================================================
# UTILITIES
# =========================================================

def get_num_parameters(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def make_optimizer(optimizer_name: str, model: nn.Module, lr: float = 0.001):
    """
    Build optimizer with default PyTorch settings.
    Exception: Momentum+Nesterov must explicitly enable momentum/nesterov,
    otherwise it is not actually that optimizer variant.
    """
    if optimizer_name == "SGD":
        return torch.optim.SGD(model.parameters())
    elif optimizer_name == "MomentumNesterov":
        return torch.optim.SGD(
            model.parameters(),
            lr=lr,          # same default lr as SGD
            momentum=0.9,
            nesterov=True
        )
    elif optimizer_name == "RMSprop":
        return torch.optim.RMSprop(model.parameters(), lr=lr)
    elif optimizer_name == "Adam":
        return torch.optim.Adam(model.parameters(), lr=lr)
    else:
        raise ValueError(f"Unknown optimizer: {optimizer_name}")


# =========================================================
# TRAIN / EVAL FUNCTIONS
# =========================================================

def train_one_epoch(
    model: nn.Module,
    loader: DataLoader,
    criterion: nn.Module,
    optimizer: torch.optim.Optimizer,
    device: torch.device
):
    model.train()

    running_loss = 0.0
    running_correct = 0
    running_total = 0

    start_time = time.time()

    for x, y in loader:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        logits = model(x)
        loss = criterion(logits, y)

        loss.backward()
        optimizer.step()

        batch_size = y.size(0)
        running_loss += loss.item() * batch_size
        running_correct += (logits.argmax(dim=1) == y).sum().item()
        running_total += batch_size

    epoch_loss = running_loss / running_total
    epoch_acc = running_correct / running_total
    epoch_time = time.time() - start_time

    return epoch_loss, epoch_acc, epoch_time


@torch.no_grad()
def evaluate(
    model: nn.Module,
    loader: DataLoader,
    criterion: nn.Module,
    device: torch.device
):
    model.eval()

    running_loss = 0.0
    running_correct = 0
    running_total = 0

    for x, y in loader:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        logits = model(x)
        loss = criterion(logits, y)

        batch_size = y.size(0)
        running_loss += loss.item() * batch_size
        running_correct += (logits.argmax(dim=1) == y).sum().item()
        running_total += batch_size

    epoch_loss = running_loss / running_total
    epoch_acc = running_correct / running_total

    return epoch_loss, epoch_acc


# =========================================================
# TRAIN ONE MODEL WITH W&B
# =========================================================

def train_model(
    optimizer_name: str,
    train_dataset,
    val_dataset,
    cfg: Config
):
    device = torch.device(cfg.device)

    train_loader = DataLoader(
        train_dataset,
        batch_size=cfg.batch_size,
        shuffle=True,
        num_workers=cfg.num_workers,
        pin_memory=cfg.pin_memory,
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=cfg.batch_size,
        shuffle=False,
        num_workers=cfg.num_workers,
        pin_memory=cfg.pin_memory,
    )

    model = DepthCNN(
        depth=cfg.depth,
        in_channels=cfg.in_channels,
        num_classes=cfg.num_classes,
        base_channels=cfg.base_channels,
        max_channels=cfg.max_channels,
        dropout=cfg.dropout,
        do=cfg.do,
        inputsize=cfg.inputsize,
    ).to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = make_optimizer(optimizer_name, model, cfg.lr)

    history = {
        "train_loss": [],
        "train_acc": [],
        "val_loss": [],
        "val_acc": [],
        "epoch_time_sec": [],
    }

    best_val_acc = -1.0
    best_val_loss = float("inf")
    best_epoch = -1
    best_model_state = copy.deepcopy(model.state_dict())

    num_params = get_num_parameters(model)

    run = None
    if cfg.use_wandb and cfg.wandb_mode != "disabled":
        wandb_config = asdict(cfg)
        wandb_config.update({
            "optimizer_name": optimizer_name,
            "num_parameters": num_params,
            "loss": "CrossEntropyLoss",
        })

        run = wandb.init(
            project=cfg.wandb_project,
            entity=cfg.wandb_entity,
            mode=cfg.wandb_mode,
            name=f"opt_{optimizer_name}_depth{cfg.depth}_e{cfg.epochs}",
            config=wandb_config,
            reinit=True,
            settings=wandb.Settings(init_timeout=300),
        )

        if cfg.wandb_watch:
            wandb.watch(model, log="all", log_freq=cfg.wandb_log_freq)

    print("=" * 80)
    print(f"Training model with optimizer = {optimizer_name} | depth = {cfg.depth}")
    print(f"Trainable parameters: {num_params:,}")
    print(model)
    print("=" * 80)

    for epoch in range(cfg.epochs):
        train_loss, train_acc, epoch_time = train_one_epoch(
            model, train_loader, criterion, optimizer, device
        )
        val_loss, val_acc = evaluate(
            model, val_loader, criterion, device
        )

        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)
        history["epoch_time_sec"].append(epoch_time)

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_val_loss = val_loss
            best_epoch = epoch + 1
            best_model_state = copy.deepcopy(model.state_dict())

        current_lr = optimizer.param_groups[0]["lr"]

        print(
            f"Epoch [{epoch+1:02d}/{cfg.epochs:02d}] | "
            f"train_loss={train_loss:.4f} | train_acc={train_acc:.4f} | "
            f"val_loss={val_loss:.4f} | val_acc={val_acc:.4f} | "
            f"time={epoch_time:.1f}s"
        )

        if run is not None:
            wandb.log({
                "epoch": epoch + 1,
                "train/loss": train_loss,
                "train/accuracy": train_acc,
                "val/loss": val_loss,
                "val/accuracy": val_acc,
                "train_val_gap/accuracy": train_acc - val_acc,
                "train_val_gap/loss": val_loss - train_loss,
                "epoch_time_sec": epoch_time,
                "lr": current_lr,
                "best_val_accuracy_so_far": best_val_acc,
            })

    model.load_state_dict(best_model_state)

    best_train_loss, best_train_acc = evaluate(model, train_loader, criterion, device)
    best_val_loss_eval, best_val_acc_eval = evaluate(model, val_loader, criterion, device)

    result = {
        "optimizer_name": optimizer_name,
        "depth": cfg.depth,
        "best_val_acc": best_val_acc,
        "best_val_loss": best_val_loss,
        "best_epoch": best_epoch,
        "best_train_acc": best_train_acc,
        "best_train_loss": best_train_loss,
        "best_val_acc_eval": best_val_acc_eval,
        "best_val_loss_eval": best_val_loss_eval,
        "avg_epoch_time_sec": sum(history["epoch_time_sec"]) / len(history["epoch_time_sec"]),
        "final_train_acc": history["train_acc"][-1],
        "final_val_acc": history["val_acc"][-1],
        "num_parameters": num_params,
    }

    if run is not None:
        wandb.summary["best_epoch"] = best_epoch
        wandb.summary["best_val_accuracy"] = best_val_acc
        wandb.summary["best_val_loss"] = best_val_loss
        wandb.summary["best_train_accuracy"] = best_train_acc
        wandb.summary["best_train_loss"] = best_train_loss
        wandb.summary["best_val_accuracy_eval"] = best_val_acc_eval
        wandb.summary["best_val_loss_eval"] = best_val_loss_eval
        wandb.summary["final_train_accuracy"] = history["train_acc"][-1]
        wandb.summary["final_val_accuracy"] = history["val_acc"][-1]
        wandb.summary["avg_epoch_time_sec"] = result["avg_epoch_time_sec"]
        wandb.finish()

    return model, history, result


# =========================================================
# TRAIN ALL OPTIMIZERS
# =========================================================

def train_all_optimizers(
    train_dataset,
    val_dataset,
    cfg: Config
):
    all_models = {}
    all_histories = {}
    all_results = {}

    for optimizer_name in cfg.optimizers:
        model, history, result = train_model(
            optimizer_name=optimizer_name,
            train_dataset=train_dataset,
            val_dataset=val_dataset,
            cfg=cfg
        )

        all_models[optimizer_name] = copy.deepcopy(model).cpu()
        all_histories[optimizer_name] = history
        all_results[optimizer_name] = result

    return all_models, all_histories, all_results


@torch.no_grad()
def evaluate_test_set(model: nn.Module, test_dataset, cfg: Config):
    device = torch.device(cfg.device)

    model = model.to(device)

    test_loader = DataLoader(
        test_dataset,
        batch_size=cfg.batch_size,
        shuffle=False,
        num_workers=cfg.num_workers,
        pin_memory=cfg.pin_memory,
    )

    criterion = nn.CrossEntropyLoss()
    test_loss, test_acc = evaluate(model, test_loader, criterion, device)

    return {
        "test_loss": test_loss,
        "test_acc": test_acc
    }


# =========================================================
# MAIN EXECUTION
# =========================================================

cfg = Config(
    in_channels=3,
    num_classes=10,
    lr = 1e-3,
    depth=12,
    inputsize=224,
    batch_size=64,
    epochs=100,
    base_channels=32,
    max_channels=256,
    dropout=0.0,
    do=0.5,
    num_workers=0,
    pin_memory=True,
    use_wandb=True,
    wandb_project="MPW-CNN",
    wandb_entity="MSE_DeLearn_SPR26",
    wandb_mode="online",
    wandb_watch=False,
)

all_models, all_histories, all_results = train_all_optimizers(
    train_dataset=train_dataset,
    val_dataset=val_dataset,
    cfg=cfg
)

print("\n" + "=" * 100)
print("SUMMARY")
print("=" * 100)

for optimizer_name, result in all_results.items():
    print(
        f"Optimizer {optimizer_name:>18} | "
        f"depth={result['depth']:>2} | "
        f"params={result['num_parameters']:,} | "
        f"best_val_acc={result['best_val_acc']:.4f} | "
        f"best_val_loss={result['best_val_loss']:.4f} | "
        f"best_train_acc={result['best_train_acc']:.4f} | "
        f"best_train_loss={result['best_train_loss']:.4f} | "
        f"best_epoch={result['best_epoch']} | "
        f"final_train_acc={result['final_train_acc']:.4f} | "
        f"final_val_acc={result['final_val_acc']:.4f} | "
        f"avg_epoch_time={result['avg_epoch_time_sec']:.2f}s"
    )

best_optimizer = max(all_results.keys(), key=lambda k: all_results[k]["best_val_acc"])
best_model = all_models[best_optimizer]

print("\nBest optimizer based on validation accuracy:")
print(f"Optimizer = {best_optimizer}")
print(f"Depth = {cfg.depth}")
print(f"Best validation accuracy = {all_results[best_optimizer]['best_val_acc']:.4f}")

# Optional test evaluation
# test_metrics = evaluate_test_set(best_model, test_dataset, cfg)
# print("\nTest set performance:")
# print(test_metrics)


# =========================================================
# OPTIONAL: W&B SUMMARY RUN
# =========================================================

if cfg.use_wandb and cfg.wandb_mode != "disabled":
    run = wandb.init(
        project=cfg.wandb_project,
        entity=cfg.wandb_entity,
        mode=cfg.wandb_mode,
        name=f"optimizer_comparison_depth{cfg.depth}",
        reinit=True,
        settings=wandb.Settings(init_timeout=300),
    )

    comparison_table = wandb.Table(columns=[
        "optimizer_name",
        "depth",
        "num_parameters",
        "best_val_acc",
        "best_val_loss",
        "best_epoch",
        "best_train_acc",
        "best_train_loss",
        "final_train_acc",
        "final_val_acc",
        "avg_epoch_time_sec",
    ])

    for optimizer_name, result in all_results.items():
        comparison_table.add_data(
            result["optimizer_name"],
            result["depth"],
            result["num_parameters"],
            result["best_val_acc"],
            result["best_val_loss"],
            result["best_epoch"],
            result["best_train_acc"],
            result["best_train_loss"],
            result["final_train_acc"],
            result["final_val_acc"],
            result["avg_epoch_time_sec"],
        )

    wandb.log({
        "optimizer_comparison_table": comparison_table,
        "best_optimizer": best_optimizer,
        "best_optimizer_val_acc": all_results[best_optimizer]["best_val_acc"],
        "depth": cfg.depth,
    })

    wandb.summary["best_optimizer"] = best_optimizer
    wandb.summary["best_optimizer_val_acc"] = all_results[best_optimizer]["best_val_acc"]
    wandb.summary["depth"] = cfg.depth
    wandb.finish()

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from C:\Users\lucas\_netrc.
wandb: Currently logged in as: lucas-j-keller98 (MSE_DeLearn_SPR26) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


Training model with optimizer = SGD | depth = 12
Trainable parameters: 3,537,130
DepthCNN(
  (features): Sequential(
    (0): ConvBlock(
      (block): Sequential(
        (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (2): ReLU(inplace=True)
      )
    )
    (1): ConvBlock(
      (block): Sequential(
        (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (2): ReLU(inplace=True)
        (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
      )
    )
    (2): ConvBlock(
      (block): Sequential(
        (0): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (2)

best_val_accuracy_so_far,▁▂▂▃▄▆▆▆▇▇▇▇▇▇▇█████████████████████████
epoch,▁▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▄▅▅▅▅▅▅▅▆▆▆▆▆▆▆▆▇▇▇███
epoch_time_sec,▆▄▄▃▄▄▃▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▂▁▁▁▁▁▁▁▁▁▁█
lr,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train/accuracy,▁▂▃▄▄▅▅▅▅▆▆▆▆▇▇▇▇▇██████████████████████
train/loss,█▆▆▅▅▅▄▄▄▄▄▃▃▃▃▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train_val_gap/accuracy,▁▂▂▃▂▃▃▃▃▄▃▄▆▃▄▅▄▄▆▅▅▅▅▆▄▆▄▅▅▄▅▆▄▅▅▅█▅▅▅
train_val_gap/loss,▁▁▁▁▁▁▁▁▁▁▂▂▂▂▃▂▂▂▂▃▅▃▃▃▃▃▃▃▃▃▃▃▃▅▃▃▃▃█▃
val/accuracy,▁▄▄▅▅▆▆▇▆▆▆▃▆█▇▇▆▇▇▇▇▆▇▅█▇██▅█▇█▇██▃█▇▇█
val/loss,▄▄▃▃▃▃▃▂▂▄▃█▁▂▂▂▂▅▂▂▁▃▄▂▁▂▃▂▁▁▃▁▁▂▂▂▃▂▁▁
avg_epoch_time_sec,47.99837


Training model with optimizer = MomentumNesterov | depth = 12
Trainable parameters: 3,537,130
DepthCNN(
  (features): Sequential(
    (0): ConvBlock(
      (block): Sequential(
        (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (2): ReLU(inplace=True)
      )
    )
    (1): ConvBlock(
      (block): Sequential(
        (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (2): ReLU(inplace=True)
        (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
      )
    )
    (2): ConvBlock(
      (block): Sequential(
        (0): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True

best_val_accuracy_so_far,▁▁▁▄▄▆▆▆▆▆▆▆▆███████████████████████████
epoch,▁▁▁▂▂▂▂▂▂▃▃▃▃▃▃▃▄▄▅▅▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇▇▇▇█
epoch_time_sec,█▇█▇█▂▁▂▁▁▁▂▂▂▂▂▁▁▂▂▁▁▁▁▂▂▂▁▂▁▁▁▁▁▁▁▁▁▁▃
lr,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train/accuracy,▁▂▃▄▅▇▇█████████████████████████████████
train/loss,█▆▅▄▃▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train_val_gap/accuracy,▁▂▁▂▆▆▅▆▅█▆▅▆▆▅▅▅▆▅▅▅▆▇▅▅▅▅▅▅▅▅▅▅▅▅▅▅▅▅▅
train_val_gap/loss,▁▁▂▃▃▃▂█▅▄▃▃▃▃▃▃▃▃▅▃▃▃▃▃▃▃▂▃▂▃▃▃▃▃▃▃▃▃▃▃
val/accuracy,▂▂▂▄▅▁▁▃▅▇▆▇██▇█▇███▇▄▇▇█████████▇██████
val/loss,▅▃▄▄▆▆▃█▄▂▂▁▂▁▂▁▁▁▁▁▁▂▄▃▂▁▂▁▁▁▁▁▁▁▁▁▁▁▁▁
avg_epoch_time_sec,49.32156


...

best_val_accuracy_so_far,▁▄▅▆▆▇▇▇▇▇██████████████████████████████
epoch,▁▁▂▂▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▅▅▆▆▆▆▇▇▇▇▇▇██
epoch_time_sec,█▅▅▅▅▅▅▆▅▄▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▃▆▅▅▅▆▁▁▁▁▁▁▁▁
lr,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train/accuracy,▁▂▄▅▆▇▇▇▇▇██████████████████████████████
train/loss,█▃▃▃▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train_val_gap/accuracy,▁▄▄▆▅▇█▇▇▇▇█▇▇█▆▇▇▇▇██▇▇▇█▇▇▇▇▇█▇▇█▇▇▇▇▇
train_val_gap/loss,▁▁▂█▂▂▂▂▂▃▃▄▃▃▃▄▄▄▄▃▄▃▄▄▄▅▄▄▄▅▄▄▅▄▄▄▃▄▄▄
val/accuracy,▁▂▃▃▅▆▇▅▆▆▆▇▇▇▆██▇███▇█▇▇██▇▇██▇█▇██████
val/loss,█▆▅▄▃▂▁▄▃▂▂▁▂▄▂▃▁▁▄▁▂▃▃▁▃▃▃▂▂▂▃▂▃▃▄▂▂▂▂▂
avg_epoch_time_sec,49.47671


Training model with optimizer = Adam | depth = 12
Trainable parameters: 3,537,130
DepthCNN(
  (features): Sequential(
    (0): ConvBlock(
      (block): Sequential(
        (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (2): ReLU(inplace=True)
      )
    )
    (1): ConvBlock(
      (block): Sequential(
        (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (2): ReLU(inplace=True)
        (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
      )
    )
    (2): ConvBlock(
      (block): Sequential(
        (0): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (2

best_val_accuracy_so_far,▁▃▄▄▄▆▆▆▆▇▇▇▇▇▇▇▇▇██████████████████████
epoch,▁▁▁▂▂▂▂▂▃▃▃▃▃▃▃▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇█████
epoch_time_sec,▅▅▅▅▅▅▅▅▅▅▅▅▅▆▅▅▅▅▅▅▅▅▅▅▅▅▅▅▅▅▅▅▅▅▅▇▁▃█▅
lr,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train/accuracy,▁▃▄▆▆▇▇▇████████████████████████████████
train/loss,█▆▅▄▃▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train_val_gap/accuracy,▂▂▁▂▃▄▄▅▆▅▇▆▇▇█▇▆▆▇▆▇▇▆▆▇▇▆▇▆▆▆▆▆▆▆▆▆▆▆▆
train_val_gap/loss,▁▂▄▅▅▅▇▇▇▇▇▇▇▇█▇▆██▇█▇▇▇▇▇▆▇▇▇▇▇▇▆▇▇▇▇█▇
val/accuracy,▁▄▄▅▅▆▇▇▇▇▇▇▇▇▇▇█▇█▇█████████▇██▇███████
val/loss,█▄▃▃▂▁▃▂▃▁▂▄▁▂▂▂▁▂▃▂▂▂▂▁▂▂▂▁▁▂▁▁▁▃▂▂▂▁▂▂
avg_epoch_time_sec,48.87242



SUMMARY
Optimizer                SGD | depth=12 | params=3,537,130 | best_val_acc=0.7590 | best_val_loss=0.7766 | best_train_acc=1.0000 | best_train_loss=0.0084 | best_epoch=82 | final_train_acc=0.9996 | final_val_acc=0.7552 | avg_epoch_time=48.00s
Optimizer   MomentumNesterov | depth=12 | params=3,537,130 | best_val_acc=0.8757 | best_val_loss=0.4446 | best_train_acc=1.0000 | best_train_loss=0.0001 | best_epoch=73 | final_train_acc=1.0000 | final_val_acc=0.8708 | avg_epoch_time=49.32s
Optimizer            RMSprop | depth=12 | params=3,537,130 | best_val_acc=0.8803 | best_val_loss=0.5834 | best_train_acc=0.9997 | best_train_loss=0.0015 | best_epoch=87 | final_train_acc=0.9966 | final_val_acc=0.8650 | avg_epoch_time=49.48s
Optimizer               Adam | depth=12 | params=3,537,130 | best_val_acc=0.8767 | best_val_loss=0.6495 | best_train_acc=0.9995 | best_train_loss=0.0015 | best_epoch=94 | final_train_acc=0.9953 | final_val_acc=0.8697 | avg_epoch_time=48.87s

Best optimizer based on va

best_optimizer_val_acc,▁
depth,▁
best_optimizer,RMSprop
best_optimizer_val_acc,0.88033
depth,12
